# T2 prep 1: installer files (wheels) for the offline run

The offline run (`t2_disasteller_local_run.ipynb`) can't download anything. So this notebook
downloads, once, the installer files ("wheels") of everything that run must install:

- **vLLM 0.25.1**: the version that ran the team's 7B on two T4 GPUs in Abrar's `3_vllm_evaluation.ipynb`;
- **CrewAI 1.15.2**: the version DisasTeller was tested with on the laptop (3 Nov 2026);
- and every package they need.

**Settings:** Internet **on**. Accelerator: None is enough.

**Then:** Save Version → **Save & Run All (Commit)**. Its output folder `wheels/` becomes an input
of the run notebook (**+ Add Input → Your Work** → this notebook).

In [ ]:
import glob, json, os, subprocess, sys, time

PACKAGES = ["vllm==0.25.1", "crewai==1.15.2"]
WHEELS = "/kaggle/working/wheels"
os.makedirs(WHEELS, exist_ok=True)
print("Python", sys.version.split()[0])


def pip(*args):
    """Run pip, keep its output in pip_log.txt, and stop on an error."""
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    with open("/kaggle/working/pip_log.txt", "a") as f:
        f.write(f"\n$ pip {' '.join(args)}\n{r.stdout}\n{r.stderr}\n")
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"pip {args[0]} failed: see pip_log.txt")
    return r.stdout

Download the packages and everything they need (several GB, a few minutes).

In [ ]:
t = time.time()
pip("download", "--dest", WHEELS, *PACKAGES)
files = sorted(os.listdir(WHEELS))
size = sum(os.path.getsize(os.path.join(WHEELS, f)) for f in files)
print(f"{len(files)} files, {size / 1e9:.1f} GB, in {(time.time() - t) / 60:.1f} min")

A few packages may come only as source code. Building them needs the internet, so build them into wheels now.

In [ ]:
sources = [f for f in glob.glob(f"{WHEELS}/*") if f.endswith((".tar.gz", ".zip"))]
for src in sources:
    pip("wheel", "--no-deps", "--wheel-dir", WHEELS, src)
    os.remove(src)
print(f"Built {len(sources)} wheels from source:", [os.path.basename(s) for s in sources])

**The test:** can pip install everything from this folder alone, with no internet?
`--ignore-installed` pretends nothing is installed yet, so the test doesn't lean on what this Kaggle image happens to have.

In [ ]:
pip("install", "--dry-run", "--ignore-installed", "--no-index", "--find-links", WHEELS,
    "--report", "/kaggle/working/install_report.json", *PACKAGES)
report = json.load(open("/kaggle/working/install_report.json"))
names = sorted(f"{p['metadata']['name']}=={p['metadata']['version']}" for p in report["install"])
print(f"OK: pip can install all {len(names)} packages from this folder, with no internet.")
print([n for n in names if n.split("==")[0].lower() in ("vllm", "crewai", "torch", "transformers", "openai", "pydantic")])

In [ ]:
files = sorted(os.listdir(WHEELS))
manifest = {
    "packages": PACKAGES,
    "python": sys.version.split()[0],
    "made": time.strftime("%Y-%m-%d %H:%M:%S"),
    "would_install": names,
    "files": {f: os.path.getsize(os.path.join(WHEELS, f)) for f in files},
}
with open(f"{WHEELS}/t2_wheels_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Done: {len(files)} files, {sum(manifest['files'].values()) / 1e9:.1f} GB, and t2_wheels_manifest.json")